# Baseline Experiment - Binary Classification - Llama-3.2-3B

Apply Llama-3.2-3B to classify whether a tweet is relevant to a disater or not, in other word, binary classification.

## A. Setup

In [ ]:
%pip install transformers torch tqdm datasets accelerate huggingface_hub python-dotenv

In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

from dotenv import load_dotenv
load_dotenv()

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, TextStreamer
from datasets import Dataset, load_from_disk, load_dataset

import configuration
from src import setup, data_utils
# from src.models import bert


/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.2.0)/charset_normalizer (None) doesn't match a supported version!
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from huggingface_hub import login
login(os.getenv("HF_TOKEN"))

base_model = "meta-llama/Llama-3.2-3B"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## B. Data

### B.1. Load sets

In [11]:
data_frac = data_utils.DATA_FRACTION

df_train, df_val, df_test = data_utils.load_datasets()

### B.2. Shrink dataset size for development purpose

In [ ]:
# # Comment out this cell to use the full dataset. This is just for quick testing.
# train_size = 100
# val_size = int(train_size * len(df_val) / len(df_train))
# test_size = int(train_size * len(df_test) / len(df_train))

# df_train = df_train.sample(n=train_size, random_state=setup.RANDOM_SEED)
# df_val = df_val.sample(n=val_size, random_state=setup.RANDOM_SEED)
# df_test = df_test.sample(n=test_size, random_state=setup.RANDOM_SEED)

In [ ]:
# Load data as Hugging Face Datasets
ds_train, ds_val, ds_test = bert.create_datasets(df_train, df_val, df_test)

## C. Tokenization

In [3]:
# https://huggingface.co/google-bert/bert-base-uncased
# bert-base-uncased - 110M
tokenizer = AutoTokenizer.from_pretrained(base_model)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-3B.
403 Client Error. (Request ID: Root=1-6a06bf38-3ede24c57e8fd1aa66e58559;fc8ec3bd-7be5-47d0-9163-f1e30ad821dd)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-3B/resolve/main/config.json.
Your request to access model meta-llama/Llama-3.2-3B is awaiting a review from the repo authors.

### C.1. Quick preview on dataset to find optimal parameter for `tokenizer`

In [ ]:
bert.max_length_dist(df_train, 'tweet_text', tokenizer)

So, set the `max_length` to `64`.

### C.2. Do the Tokenization

In [ ]:
save_path = Path(f"../tokens/BERT/{data_frac}")
train_tokenized, val_tokenized, test_tokenized = bert.load_or_tokenize(
    ds_train, ds_val, ds_test, tokenizer, save_path
    , force_retokenize=True
)

## B. Fine-tuning BERT

### B.0. Shrink dataset size for development purpose

### B.1. Preparation

In [ ]:
device = setup.setup_device_with_seeds()

batch_size = 16
learning_rate = 5e-5
num_epochs = 5
patience = 2  # early stopping, if validation loss does not improve for this many epochs

train_loader = DataLoader(train_tokenized, batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(val_tokenized, batch_size=batch_size)

# Optimizer
bert_base = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2  # e.g., binary sentiment
)
bert_base.config.problem_type = "single_label_classification"
optimizer = AdamW(bert_base.parameters(), lr=learning_rate)

configs = {
    "batch_size": batch_size,
    "bert": bert_base,
    "optimizer": optimizer,
    "device": device,
    "num_epochs": num_epochs,
    "patience": patience
}

### B.2. Fine-tune

In [ ]:
model, train_loss_history, val_loss_history, val_acc_history = bert.finetune(train_tokenized, val_tokenized, configs)

In [ ]:
data_utils.plot_fine_tune_history(train_loss_history, val_loss_history, val_acc_history)

### B.3. Predict on the Test set

In [ ]:
predictions = bert.predict(model, test_tokenized, device)

In [ ]:
bert.report_metrics(test_tokenized, predictions)

In [ ]:
data_utils.group_report_metrics(df_test, predictions, group_by="subset", labels="informative")